In [2]:
import os
import re
import pandas as pd

root_dir = "checkpoints/bert4rec"  # 수정 필요 시 여기에 경로 설정

val_pattern = re.compile(
    r"Val Loss: ([\d.]+), HR@1: ([\d.]+), HR@5: ([\d.]+), HR@10: ([\d.]+), "
    r"NDCG@5: ([\d.]+), NDCG@10: ([\d.]+), MRR: ([\d.]+)"
)
test_pattern = re.compile(
    r"Test Loss: ([\d.]+), HR@1: ([\d.]+), HR@5: ([\d.]+), HR@10: ([\d.]+), "
    r"NDCG@5: ([\d.]+), NDCG@10: ([\d.]+), MRR: ([\d.]+)"
)

results = []
for folder in os.listdir(root_dir):
    subdir = os.path.join(root_dir, folder)
    if not os.path.isdir(subdir):
        continue
    val_path = os.path.join(subdir, "result_val.txt")
    test_path = os.path.join(subdir, "result_test.txt")
    if os.path.exists(val_path) and os.path.exists(test_path):
        with open(val_path) as f: val_txt = f.read()
        with open(test_path) as f: test_txt = f.read()
        val_match = val_pattern.search(val_txt)
        test_match = test_pattern.search(test_txt)
        if val_match and test_match:
            val_metrics = list(map(float, val_match.groups()))
            test_metrics = list(map(float, test_match.groups()))
            results.append([folder] + val_metrics + test_metrics)

columns = [
    "folder", 
    "val_loss", "val_hr@1", "val_hr@5", "val_hr@10", "val_ndcg@5", "val_ndcg@10", "val_mrr",
    "test_loss", "test_hr@1", "test_hr@5", "test_hr@10", "test_ndcg@5", "test_ndcg@10", "test_mrr"
]
df = pd.DataFrame(results, columns=columns)
df = df.sort_values(by="val_loss")
print(df.head(5))  # 상위 5개 결과 출력


                              folder  val_loss  val_hr@1  val_hr@5  val_hr@10  \
28  head_4_layers_4_batch_256_seed_0    3.0052    0.2505    0.5984     0.7413   
20  head_8_layers_2_batch_256_seed_0    3.0157    0.2534    0.6008     0.7409   
16  head_2_layers_4_batch_128_seed_0    3.0235    0.2450    0.6020     0.7413   
38  head_8_layers_8_batch_256_seed_0    3.0252    0.2559    0.5952     0.7418   
17  head_4_layers_4_batch_128_seed_0    3.0323    0.2520    0.5939     0.7337   

    val_ndcg@5  val_ndcg@10  val_mrr  test_loss  test_hr@1  test_hr@5  \
28      0.4323       0.4785   0.4084     3.0783     0.2552     0.5698   
20      0.4348       0.4802   0.4104     3.1198     0.2539     0.5676   
16      0.4320       0.4773   0.4065     3.0695     0.2613     0.5739   
38      0.4331       0.4805   0.4110     3.1077     0.2640     0.5606   
17      0.4310       0.4763   0.4081     3.1275     0.2532     0.5579   

    test_hr@10  test_ndcg@5  test_ndcg@10  test_mrr  
28      0.7084      

In [8]:
columns_2 = [
    "folder", 
    "val_loss",
    "test_loss", "test_hr@1", "test_hr@5", "test_hr@10", 
    "test_ndcg@5", "test_ndcg@10", "test_mrr"
]
pd.set_option("display.max_columns", None)   # 컬럼 수 무제한 출력
pd.set_option("display.width", 200)          # 콘솔 너비 늘리기
pd.set_option("display.colheader_justify", "left")  # 헤더 정렬

df_2 = df[columns_2].sort_values(by="val_loss")
print(df_2.head(5))


   folder                             val_loss  test_loss  test_hr@1  test_hr@5  test_hr@10  test_ndcg@5  test_ndcg@10  test_mrr
28  head_4_layers_4_batch_256_seed_0  3.0052    3.0783     0.2552     0.5698     0.7084      0.4186       0.4636        0.4002  
20  head_8_layers_2_batch_256_seed_0  3.0157    3.1198     0.2539     0.5676     0.6970      0.4165       0.4584        0.3972  
16  head_2_layers_4_batch_128_seed_0  3.0235    3.0695     0.2613     0.5739     0.7084      0.4229       0.4664        0.4040  
38  head_8_layers_8_batch_256_seed_0  3.0252    3.1077     0.2640     0.5606     0.7059      0.4157       0.4628        0.4005  
17  head_4_layers_4_batch_128_seed_0  3.0323    3.1275     0.2532     0.5579     0.6968      0.4114       0.4565        0.3948  
